# 🧪 Javtiful → LuluStream Test
No Telegram. This notebook resolves a Javtiful media URL inside Colab and tests LuluStream remote upload. Because Javtiful URLs may be IP-locked, a remote LuluStream worker may not be able to fetch the URL. The optional direct-upload path keeps the media request inside Colab and does not bypass the IP restriction.

In [ ]:
import os,shutil,subprocess
from pathlib import Path
REPO='https://github.com/Telebotfaroff/javtiful-scraper.git'; BRANCH='javdl'; WORKDIR=Path('/content/javdl-lulustream-test')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)],check=True)
os.chdir(WORKDIR)
print('JAVDL extractor ready.')

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!python -m pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium

In [ ]:
from google.colab import userdata
LULUSTREAM_API_KEY=userdata.get('LULUSTREAM_API_KEY')
if not LULUSTREAM_API_KEY: raise RuntimeError('Add LULUSTREAM_API_KEY to Colab Secrets.')
JAVTIFUL_URL=input('Javtiful video URL: ').strip()
if not JAVTIFUL_URL.startswith(('http://','https://')): raise ValueError('Invalid URL.')
print('Credentials and URL ready.')

In [ ]:
from app.extractor.javtiful import JavtifulExtractor
extractor=JavtifulExtractor(); video=extractor.extract(JAVTIFUL_URL)
print('Title:',video.title); print('Duration:',video.duration); print('Thumbnail:',video.thumbnail)
print('\nResolved sources:')
for q,u in video.qualities.items(): print(q,'=>',u)
if not video.qualities: raise RuntimeError('No media source resolved.')
resolved_url=next((u for u in video.qualities.values() if '.mp4' in u.lower()),None) or next((u for u in video.qualities.values() if '.m3u8' in u.lower()),None) or next(iter(video.qualities.values()))
print('\nSelected resolved URL:\n',resolved_url)

In [ ]:
import requests
params={'key':LULUSTREAM_API_KEY,'url':resolved_url,'file_public':1,'file_adult':1}
r=requests.get('https://lulustream.com/api/upload/url',params=params,timeout=30)
print('Remote upload HTTP:',r.status_code); print(r.text)
try:
 data=r.json(); remote_code=(data.get('result') or {}).get('filecode')
 print('Remote file code:',remote_code or 'not returned')
except Exception: remote_code=None

## Optional direct-upload fallback
If the remote upload fails because LuluStream cannot access the IP-locked URL, this downloads the resolved source from Colab and uploads the local file to LuluStream.

In [ ]:
RUN_DIRECT=input('Run direct Colab download + LuluStream upload? [y/N]: ').strip().lower()=='y'
print('Direct upload:',RUN_DIRECT)

In [ ]:
import subprocess,shlex,requests
if not RUN_DIRECT:
 print('Skipped.')
else:
 out=Path('/content/lulustream-test.mp4')
 headers='User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36\r\nReferer: '+JAVTIFUL_URL+'\r\n'
 cmd=['ffmpeg','-y','-headers',headers,'-i',resolved_url,'-c','copy',str(out)]
 print('Downloading inside Colab...'); subprocess.run(cmd,check=True)
 print(f'Downloaded {out.stat().st_size/1048576:.2f} MB')
 sr=requests.get('https://lulustream.com/api/upload/server',params={'key':LULUSTREAM_API_KEY},timeout=30); sr.raise_for_status(); server=sr.json().get('result')
 print('Upload server:',server)
 with out.open('rb') as fh:
  ur=requests.post(server,data={'key':LULUSTREAM_API_KEY,'file_title':video.title or out.stem,'file_public':'1','file_adult':'1'},files={'file':(out.name,fh,'video/mp4')},timeout=3600)
 print('Direct upload HTTP:',ur.status_code); print(ur.text); ur.raise_for_status()
 for item in ur.json().get('files',[]):
  code=item.get('filecode'); print('File code:',code,'status:',item.get('status')); print('Player:',f'https://lulustream.com/{code}.html' if code else '')

In [ ]:
code=input('LuluStream file code to verify (blank to skip): ').strip()
if code:
 r=requests.get('https://lulustream.com/api/file/info',params={'key':LULUSTREAM_API_KEY,'file_code':code},timeout=30)
 print(r.status_code); print(r.text)
else: print('Skipped.')